# RREF v6e measured run

Colab notebook for the large RREF 8x8/F_101 measured run. It runs the tracked harness `scripts/rref_measured_run.py` with the `colab-v6e1-large` profile and writes only compact JSON plus Markdown under `/tmp`.

Use a TPU v6e runtime. The harness asserts JAX backend `tpu` before training; if TPU is unavailable, it stops before generating shards or checkpoints.


## Runtime boundary

- Runtime: Python 3.12.
- Hardware accelerator: TPU v6e.
- Heavy artifacts stay in `/tmp/nf-rref-colab-v6e1-large`.
- Download only `/tmp/rref_8x8_mod101_colab_v6e1_large.json` and `/tmp/rref_8x8_mod101_colab_v6e1_large.md`.
- Do not export or commit Colab PDFs, NPZ shards, checkpoints, or raw logs.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/JJYYY-JJY/tpu-verifier-normalforms.git"
REPO_ROOT = Path("/content/tpu-verifier-normalforms")
WORK_DIR = Path("/tmp/nf-rref-colab-v6e1-large")
OUT_JSON = Path("/tmp/rref_8x8_mod101_colab_v6e1_large.json")
OUT_MD = Path("/tmp/rref_8x8_mod101_colab_v6e1_large.md")


def run_cmd(command: list[str], *, cwd: Path | None = None, redact: str | None = None) -> str:
    display_command = [part.replace(redact, "***") if redact else part for part in command]
    print("$", " ".join(display_command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}")
    return completed.stdout


def clone_repo() -> None:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    try:
        run_cmd(["git", "clone", "--branch", "main", "--depth", "1", REPO_URL, str(REPO_ROOT)])
        return
    except RuntimeError:
        token = getpass("GitHub token with repo read access; input is hidden: ")
        if not token:
            raise
        auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        run_cmd(["git", "clone", "--branch", "main", "--depth", "1", auth_url, str(REPO_ROOT)], redact=token)
        run_cmd(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_ROOT)


print("Python", platform.python_version())
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("nf-agent requires Python 3.12; switch the Colab runtime before continuing")


## Checkout and install

The install cell uses the repo's Python 3.12 package contract. If a stale checkout exists in `/content/tpu-verifier-normalforms`, it is replaced with the current `main` branch. If public clone fails because the repo is private, the cell asks for a hidden GitHub token with repo read access and resets `origin` back to the non-token URL after cloning.


In [ ]:
clone_repo()
run_cmd([sys.executable, "-m", "pip", "install", "-U", "pip"])
run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements-dev.txt", "-e", "."], cwd=REPO_ROOT)
run_cmd(["nf-agent", "--help"], cwd=REPO_ROOT)


## TPU backend check

This must print `tpu`. If it does not, switch the Colab runtime to TPU v6e before running the measured profile.


In [ ]:
os.environ["JAX_PLATFORMS"] = "tpu,cpu"
probe_env = os.environ.copy()
probe_env["JAX_PLATFORMS"] = "tpu,cpu"
probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import json\nimport jax\nprint(json.dumps({\"backend\": jax.default_backend(), \"local_device_count\": jax.local_device_count(), \"devices\": [str(d) for d in jax.devices()]}))",
    ],
    text=True,
    capture_output=True,
    check=False,
    env=probe_env,
)
if probe.stderr:
    print(probe.stderr, file=sys.stderr)
if probe.returncode != 0:
    raise RuntimeError(f"TPU backend probe failed with exit code {probe.returncode}")

probe_payload = json.loads(probe.stdout)
backend = probe_payload["backend"]
print("JAX backend:", backend)
print("local_device_count:", probe_payload["local_device_count"])
print("devices:", probe_payload["devices"])
if backend != "tpu":
    raise RuntimeError(f"expected TPU backend, got {backend!r}")


## Run measured profile

This runs shard generation, batch calibration, 2000 training steps, and a 512-sample compact RREF benchmark. It may take several minutes. The command fails before training if backend assertion fails.


In [ ]:
for path in (OUT_JSON, OUT_MD):
    if path.exists():
        path.unlink()

run_cmd(
    [
        sys.executable,
        "scripts/rref_measured_run.py",
        "--profile",
        "colab-v6e1-large",
        "--work-dir",
        str(WORK_DIR),
        "--out",
        str(OUT_JSON),
        "--summary-md",
        str(OUT_MD),
    ],
    cwd=REPO_ROOT,
)


## Inspect compact result

The JSON should have `schema_version: rref-measured-run-v1`, `profile: colab-v6e1-large`, backend `tpu`, selected batch size, training metrics, and compact benchmark aggregates.


In [ ]:
payload = json.loads(OUT_JSON.read_text())
print(json.dumps({
    "status": payload["status"],
    "profile": payload["profile"],
    "backend": payload["jax"]["backend"],
    "selected_batch_size": payload["selected_batch_size"],
    "train_final_loss": payload["train"]["final_loss"],
    "checkpoint_step": payload["train"]["checkpoint_step"],
    "leftmost_status_counts": payload["benchmark"]["policies"]["leftmost"]["status_counts"],
    "neural_status_counts": payload["benchmark"]["policies"]["neural"]["status_counts"],
}, indent=2, sort_keys=True))
print("JSON:", OUT_JSON)
print("Markdown:", OUT_MD)


## Download compact artifacts

Download both files and place them in the repo as:

- `results/measured/rref_8x8_mod101_colab_v6e1_large.json`
- `results/measured/rref_8x8_mod101_colab_v6e1_large.md`


In [ ]:
from google.colab import files

files.download(str(OUT_JSON))
files.download(str(OUT_MD))
